In [1]:
# Cell 1 — environment + GPU check (right-sizing to 48GB)
!rocm-smi
!rocm-smi --showproductname
print("=" * 60)
import torch
print("PyTorch:", torch.__version__)
print("GPU visible to PyTorch:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
    free, total = torch.cuda.mem_get_info()
    print(f"VRAM total: {total/1e9:.1f} GB | free now: {free/1e9:.1f} GB")
try:
    import vllm
    print("vLLM:", vllm.__version__)
except Exception as e:
    print("vLLM import problem:", e)
import os
print("HF token present:", bool(os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")))



======================================== ROCm System Management Interface ========================================
================================================== Concise Info ==================================================
Device  Node  IDs              Temp    Power  Partitions          SCLK  MCLK   Fan    Perf  PwrCap  VRAM%  GPU%  
              (DID,     GUID)  (Edge)  (Avg)  (Mem, Compute, ID)                                                 
0       2     0x744b,   6853   29.0°C  7.0W   N/A, N/A, 0         0Mhz  96Mhz  20.0%  auto  241.0W  0%     0%    
============================================== End of ROCm SMI Log ===============================================


============================ ROCm System Management Interface ============================
====================================== Product Info ======================================
GPU[0]		: get_name, Error when calling libdrm
GPU[0]		: Card Series: 		N/A
GPU[0]		: Card Model: 		0x744b
GPU[0]		: Card Vendor

In [2]:
# Cell 2 — HF login (token stays hidden, never written into the notebook)
from huggingface_hub import login, whoami, model_info
from getpass import getpass
login(getpass("Paste HF token (input hidden): "))
print("Logged in as:", whoami()["name"])

# Confirm access to the gated repo BEFORE downloading ~24GB
try:
    model_info("google/gemma-3-12b-it")
    print("Access to google/gemma-3-12b-it: OK")
except Exception as e:
    print("ACCESS PROBLEM — accept the license on the model page first:")
    print(str(e)[:200])

/opt/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Paste HF token (input hidden):  ········


Logged in as: exnav29
Access to google/gemma-3-12b-it: OK


In [3]:
# Cell 3 — load a right-sized Gemma on the AMD GPU via vLLM
import time, torch
from vllm import LLM, SamplingParams

t0 = time.time()
llm = LLM(
    model="google/gemma-3-12b-it",
    dtype="bfloat16",
    max_model_len=8192,          # caps KV-cache; plenty for a study guide
    gpu_memory_utilization=0.90,
)
load_s = time.time() - t0
print(f"\n✅ Gemma loaded in {load_s:.0f}s")

free, total = torch.cuda.mem_get_info()
print(f"VRAM in use after load: {(total-free)/1e9:.1f} GB of {total/1e9:.1f} GB")
print("=" * 60)

INFO 07-08 11:28:09 [utils.py:223] non-default args: {'dtype': 'bfloat16', 'max_model_len': 8192, 'disable_log_stats': True, 'model': 'google/gemma-3-12b-it'}
INFO 07-08 11:28:21 [model.py:529] Resolved architecture: Gemma3ForConditionalGeneration
INFO 07-08 11:28:21 [model.py:1549] Using max model len 8192


2026-07-08 11:28:21,768	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


INFO 07-08 11:28:21 [scheduler.py:224] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 07-08 11:28:21 [vllm.py:689] Asynchronous scheduling is enabled.


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


WARNING 07-08 11:28:47 [system_utils.py:140] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
(EngineCore_DP0 pid=947) INFO 07-08 11:28:53 [core.py:97] Initializing a V1 LLM engine (v0.16.1.dev0+g89a77b108.d20260318) with config: model='google/gemma-3-12b-it', speculative_config=None, tokenizer='google/gemma-3-12b-it', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=8192, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=True, quantization=None, enforce_eager=False, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_f

...

(EngineCore_DP0 pid=947) INFO 07-08 11:29:11 [gpu_model_runner.py:4124] Starting to load model google/gemma-3-12b-it...
(EngineCore_DP0 pid=947) INFO 07-08 11:29:11 [rocm.py:430] Using Flash Attention (Triton backend) for ViT model on RDNA.
(EngineCore_DP0 pid=947) INFO 07-08 11:29:11 [mm_encoder_attention.py:77] Using AttentionBackendEnum.FLASH_ATTN for MMEncoderAttention.
(EngineCore_DP0 pid=947) WARNING 07-08 11:29:11 [activation.py:643] [ROCm] PyTorch's native GELU with tanh approximation is unstable. Falling back to GELU(approximate='none').
(EngineCore_DP0 pid=947) INFO 07-08 11:29:11 [vllm.py:689] Asynchronous scheduling is enabled.
(EngineCore_DP0 pid=947) INFO 07-08 11:29:12 [rocm.py:377] Using Triton Attention backend.
(EngineCore_DP0 pid=947) WARNING 07-08 11:29:12 [activation.py:279] [ROCm] PyTorch's native GELU with tanh approximation is unstable with torch.compile. For native implementation, fallback to 'none' approximation. The custom kernel implementation is unaffected.

Loading safetensors checkpoint shards:   0% Completed | 0/5 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  20% Completed | 1/5 [00:02<00:08,  2.19s/it]
Loading safetensors checkpoint shards:  40% Completed | 2/5 [00:04<00:06,  2.33s/it]
Loading safetensors checkpoint shards:  60% Completed | 3/5 [00:07<00:04,  2.36s/it]
Loading safetensors checkpoint shards:  80% Completed | 4/5 [00:09<00:02,  2.39s/it]
Loading safetensors checkpoint shards: 100% Completed | 5/5 [00:11<00:00,  2.34s/it]
Loading safetensors checkpoint shards: 100% Completed | 5/5 [00:11<00:00,  2.34s/it]
(EngineCore_DP0 pid=947) 


(EngineCore_DP0 pid=947) INFO 07-08 11:30:34 [default_loader.py:293] Loading weights took 11.97 seconds
(EngineCore_DP0 pid=947) INFO 07-08 11:30:35 [gpu_model_runner.py:4221] Model loading took 25.41 GiB memory and 82.776223 seconds
(EngineCore_DP0 pid=947) INFO 07-08 11:30:35 [gpu_model_runner.py:5140] Encoder cache will be initialized with a budget of 8192 tokens, and profiled with 32 image items of the maximum feature size.
(EngineCore_DP0 pid=947) INFO 07-08 11:30:53 [backends.py:916] Using cache directory: /root/.cache/vllm/torch_compile_cache/d177c2569c/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=947) INFO 07-08 11:30:53 [backends.py:976] Dynamo bytecode transform time: 7.05 s
(EngineCore_DP0 pid=947) INFO 07-08 11:31:24 [backends.py:351] Cache the graph of compile range (1, 8192) for later use
(EngineCore_DP0 pid=947) INFO 07-08 11:32:04 [backends.py:368] Compiling a graph for compile range (1, 8192) takes 67.22 s
(EngineCore_DP0 pid=947) INFO 07-08 11:32:04 

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:07<00:00,  6.61it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:14<00:00,  2.48it/s]


(EngineCore_DP0 pid=947) INFO 07-08 11:32:30 [gpu_model_runner.py:5246] Graph capturing finished in 23 secs, took 5.39 GiB
(EngineCore_DP0 pid=947) INFO 07-08 11:32:30 [core.py:278] init engine (profile, create kv cache, warmup model) took 115.53 seconds
INFO 07-08 11:32:31 [llm.py:355] Supported tasks: ['generate']

✅ Gemma loaded in 262s
VRAM in use after load: 49.2 GB of 51.5 GB


In [4]:
!rocm-smi



======================================== ROCm System Management Interface ========================================
================================================== Concise Info ==================================================
Device  Node  IDs              Temp    Power  Partitions          SCLK  MCLK   Fan    Perf  PwrCap  VRAM%  GPU%  
              (DID,     GUID)  (Edge)  (Avg)  (Mem, Compute, ID)                                                 
0       2     0x744b,   6853   28.0°C  16.0W  N/A, N/A, 0         0Mhz  96Mhz  20.0%  auto  241.0W  84%    0%    
============================================== End of ROCm SMI Log ===============================================


In [5]:
# Cell 4 — generate Akuapem Twi devotionals from the four passages (+ benchmark)
import time, torch

PASSAGES = {
 "Ruth 1:1-5": {
  "english": "During the time before kings ruled Israel, there was a famine. There was a man whose name was Elimelech. His wife's name was Naomi, and his sons' names were Mahlon and Chilion. They were from Bethlehem in Judah. Because of the famine they went to live for a while in Moab. While there, Elimelech died, and Naomi had only her two sons. They married women from Moab, named Orpah and Ruth. After about ten years, Mahlon and Chilion also died, so Naomi had no husband and no sons.",
  "twi": "Bere a na atemmufo di Israel so no, ɔkɔm baa asase no so. Ɔbarima bi a ofi Betlehem wɔ Yuda ne ne yere ne ne mmabarima baanu tutu fii asase no so kɔtenaa Moab. Na ɔbarima no din de Elimelek na ne yere nso din de Naomi. Ne mmabarima baanu no nso, na wɔn din de Mahlon ne Kilion. Wɔyɛ Efratifo a wofi Betlehem wɔ Yuda. Ɛbaa sɛ Naomi kunu Elimelek wu ma ɛkaa ne mmabarima baanu no. Wɔwarewaree Moabfo mmea a wɔn din de Orpa ne Rut. Wɔtenaa hɔ bɛyɛ mfe du no, Mahlon ne Kilion nso wuwui. Enti ɛmaa Naomi tenaa ase a wahwere ne mmabarima ne ne kunu."},
 "Jeremiah 29:9-13": {
  "english": "They are telling you lies, saying that I have given them the messages they tell you, but I have not appointed them. This is what Yahweh says: After you have been in Babylon for seventy years, I will help you and do what I promised, and enable you to return to Jerusalem. I know what I have planned for you — plans to cause things to go well for you, not to harm you, to give you the future you hope for. When you pray, I will heed you. If you earnestly seek me, you will find me.",
  "twi": "Wɔhyɛ atoro nkɔm wɔ me din mu. Mensomaa wɔn,” Awurade na ose. Nea Awurade se ni: “Sɛ mosom Babilonia mfe aduɔson a, mɛba abɛhwɛ mo na mɛma mʼadom mu bɔhyɛ sɛ mede mo bɛba ha bio no aba mu. Minim nhyehyɛe a mewɔ ma mo,” Awurade na ose, “Nhyehyɛe a ɛde mo besi yiye na ɛrenhaw mo, nhyehyɛe a ɛbɛma mo anidaso ne awiei pa. Afei mobɛfrɛ me na moaba abɛbɔ me mpae, na metie mo. Mobɛhwehwɛ me na mubehu me, sɛ mode mo koma nyinaa hwehwɛ me a."},
 "Matthew 3:16": {
  "english": "After he was baptized, Jesus immediately came up out of the water. Just then it was as though the sky was opened. Then Jesus saw God's Spirit coming down upon him in the form of a dove.",
  "twi": "Wɔbɔɔ Yesu asu wiei no, ofii asu no mu, na hwɛ, ɔsoro buei. Ohuu Onyankopɔn Honhom sɛ ɛresian ba ne so sɛ aborɔnoma."},
 "Psalm 29:8": {
  "english": "The voice of Yahweh causes the desert to shake.",
  "twi": "Awurade nne wosow sare so;"},
}

def prompt_for(ref, english, twi):
    return (f"You are helping a Ghanaian pastor prepare a short devotional in Akuapem Twi.\n"
            f"Below is {ref} in English and in the Akuapem Twi Bible (an expert translation).\n\n"
            f"Write a SHORT devotional in AKUAPEM TWI ONLY:\n"
            f"1. 2-3 warm, encouraging sentences reflecting on the passage.\n"
            f"2. Quote a short phrase from the Twi passage.\n"
            f"3. One closing sentence of prayer.\n"
            f"Match the vocabulary and register of the Twi Bible text. Write only in Akuapem Twi.\n\n"
            f"English: {english}\n\nAkuapem Twi: {twi}")

sp = SamplingParams(temperature=0.7, top_p=0.9, max_tokens=512)
convs = [[{"role": "user", "content": prompt_for(r, d["english"], d["twi"])}]
         for r, d in PASSAGES.items()]

t0 = time.time()
outs = llm.chat(convs, sp)
elapsed = time.time() - t0

refs = list(PASSAGES.keys())
samples, total_tok = [], 0
for ref, o in zip(refs, outs):
    text = o.outputs[0].text.strip()
    ntok = len(o.outputs[0].token_ids)
    total_tok += ntok
    samples.append(f"{'='*60}\n{ref}  ({ntok} tokens)\n{'='*60}\n{text}\n")
    print(samples[-1])

tps = total_tok / elapsed
print(f"\n>>> {total_tok} output tokens in {elapsed:.1f}s = {tps:.1f} tokens/sec\n")

# --- evidence files (download these) ---
with open("twi_samples.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(samples))
with open("amd_gemma_benchmark.txt", "w", encoding="utf-8") as f:
    f.write(
        "Every Tongue — AMD GPU inference benchmark\n"
        "Model: google/gemma-3-12b-it (Gemma 3, 12B, bfloat16)\n"
        "Hardware: AMD Radeon GPU, gfx1100 (RDNA3), 51.5 GB VRAM\n"
        "Stack: ROCm 7.2, vLLM 0.16.1, PyTorch 2.9.1, Triton attention backend\n"
        f"Weights resident: 25.41 GiB | Total VRAM in use: 49.2 GB / 51.5 GB\n"
        f"Model load time: 262 s\n"
        f"Generation throughput: {tps:.1f} tokens/sec ({total_tok} tokens / {elapsed:.1f}s, 4 prompts batched)\n")
print("Saved: twi_samples.txt, amd_gemma_benchmark.txt  (download both from the file browser)")

INFO 07-08 11:38:09 [hf.py:318] Detected the chat template content format to be 'openai'. You can set `--chat-template-content-format` to override this.


Processed prompts: 100%|██████████| 4/4 [00:14<00:00,  3.57s/it, est. speed input: 89.86 toks/s, output: 53.97 toks/s]

Ruth 1:1-5  (187 tokens)
Here's a short devotional in Akuapem Twi, following your instructions:

Medaase Nyame! Ɛyɛ nhyira a ɔyɛ sɛ yɛnka akatua no. Ɛbɛyɛ akyi sɛ Nyame bɛhyɛɛn yɛn, ne bɛgyeɛn so wɔ nhyira a ɔyɛ sɛ yɛnka akatua no. Ɛnna sɛ yɛnkyerɛ sɛ obi akyi no bɛkyerɛn.

"Ɛbaa sɛ Naomi kunu Elimelek wu ma ɛkaa ne mmabarima baanu no."

Nyame, bɛyɛɛ wo nkyerɛkyerɛɛ so, na bɛgyeɛn so wɔ nhyira a ɔyɛ sɛ yɛnka akatua no, Amen.

Jeremiah 29:9-13  (197 tokens)
Here's a short devotional in Akuapem Twi, following your instructions:

Da yɛn anim daa! Nyansafoɔ nyɛnɛ, Awurade nkyɛn bɛnkyɛn na ɔde bɛma yɛn. Ɛyɛ aduaba sɛ yɛn nkyɛn Awurade, na ɔde bɛma yɛn ho yɛn nkyɛn sɛ yɛn bɛyɛ adwuma.

“Nhyehyɛe a ɛde mo besi yiye na ɛrenhaw mo…”!  Awurade nkyɛn bɛma yɛn anidaso ne awiei pa.

Yɛnkyɛn Awurade, bɛma yɛn nkyɛn na yɛn bɛfrɛ wo, bɛyɛ adwuma wo nkyɛn. Amen.

Matthew 3:16  (176 tokens)
Here's a short devotional in Akuapem Twi, following your instructions:

Mepaakyɛwɔ dɛ Yesu ne dwumadie bi nkyɛn, n

In [6]:
# Cell 5 — same four passages, generated in SWAHILI (fair comparison vs Twi)
import time

PASSAGES_SW = {
 "Ruth 1:1-5": {
  "english": "During the time before kings ruled Israel, there was a famine. There was a man whose name was Elimelech. His wife's name was Naomi, and his sons' names were Mahlon and Chilion. They were from Bethlehem in Judah. Because of the famine they went to live for a while in Moab. While there, Elimelech died, and Naomi had only her two sons. They married women from Moab, named Orpah and Ruth. After about ten years, Mahlon and Chilion also died, so Naomi had no husband and no sons.",
  "target": "Wakati Waamuzi walipotawala Israeli, kulikuwa na njaa katika nchi. Mtu mmoja kutoka Bethlehemu ya Yuda, pamoja na mke wake na wanawe wawili, akaenda kuishi katika nchi ya Moabu. Mtu huyu aliitwa Elimeleki, mkewe aliitwa Naomi, na wanawe wawili waliitwa Maloni na Kilioni. Elimeleki mumewe Naomi akafa, hivyo Naomi akabaki na wanawe wawili. Nao wakaoa wanawake Wamoabu, jina la mmoja aliitwa Orpa, na wa pili aliitwa Ruthu. Baada ya kama miaka kumi, Maloni na Kilioni pia wakafa. Basi Naomi akabaki hana mume wala wanawe."},
 "Jeremiah 29:9-13": {
  "english": "They are telling you lies, saying that I have given them the messages they tell you, but I have not appointed them. This is what Yahweh says: After you have been in Babylon for seventy years, I will help you and do what I promised, and enable you to return to Jerusalem. I know what I have planned for you — plans to cause things to go well for you, not to harm you, to give you the future you hope for. When you pray, I will heed you. If you earnestly seek me, you will find me.",
  "target": "Wanawatabiria ninyi uongo kwa jina langu. Sikuwatuma,” asema Bwana. Hili ndilo asemalo Bwana: “Miaka sabini itakapotimia kwa ajili ya Babeli, nitakuja kwenu na kutimiza ahadi yangu ya rehema ya kuwarudisha mahali hapa. Kwa maana ninajua mipango niliyo nayo kwa ajili yenu, ni mipango ya kuwafanikisha na wala si ya kuwadhuru, ni mipango ya kuwapa tumaini katika siku zijazo. Kisha mtaniita, na mtakuja na kuniomba, nami nitawasikiliza. Mtanitafuta na kuniona, mtakaponitafuta kwa moyo wenu wote."},
 "Matthew 3:16": {
  "english": "After he was baptized, Jesus immediately came up out of the water. Just then it was as though the sky was opened. Then Jesus saw God's Spirit coming down upon him in the form of a dove.",
  "target": "Naye Yesu alipokwisha kubatizwa, mara alipotoka ndani ya maji, ghafula mbingu zikafunguka, akamwona Roho wa Mungu akishuka kama hua na kutulia juu yake."},
 "Psalm 29:8": {
  "english": "The voice of Yahweh causes the desert to shake.",
  "target": "Sauti ya Bwana hutikisa jangwa;"},
}

def prompt_sw(ref, english, target):
    return (f"You are helping a pastor prepare a short devotional in Swahili.\n"
            f"Below is {ref} in English and in the Swahili Bible (an expert translation).\n\n"
            f"Write a SHORT devotional in SWAHILI ONLY:\n"
            f"1. 2-3 warm, encouraging sentences reflecting on the passage.\n"
            f"2. Quote a short phrase from the Swahili passage.\n"
            f"3. One closing sentence of prayer.\n"
            f"Match the vocabulary and register of the Swahili Bible text. Write only in Swahili.\n\n"
            f"English: {english}\n\nSwahili: {target}")

sp = SamplingParams(temperature=0.7, top_p=0.9, max_tokens=512)
convs = [[{"role": "user", "content": prompt_sw(r, d["english"], d["target"])}]
         for r, d in PASSAGES_SW.items()]

t0 = time.time()
outs = llm.chat(convs, sp)
elapsed = time.time() - t0

samples, total_tok = [], 0
for ref, o in zip(PASSAGES_SW.keys(), outs):
    text = o.outputs[0].text.strip()
    ntok = len(o.outputs[0].token_ids)
    total_tok += ntok
    samples.append(f"{'='*60}\n{ref}  ({ntok} tokens)\n{'='*60}\n{text}\n")
    print(samples[-1])

print(f"\n>>> {total_tok} output tokens in {elapsed:.1f}s = {total_tok/elapsed:.1f} tokens/sec\n")
with open("swahili_samples.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(samples))
print("Saved: swahili_samples.txt")

Processed prompts: 100%|██████████| 4/4 [00:11<00:00,  2.87s/it, est. speed input: 101.97 toks/s, output: 45.93 toks/s]

Ruth 1:1-5  (129 tokens)
Here's a short devotional in Swahili, following your guidelines:

Mungu wetu ni mwema hata katika nyakati ngumu. Tunamkumbuka Naomi, aliyekuwa na uzoefu wa kupoteza sana, lakini bado alimwite Mungu. Hii ni ishara ya imani na tumaini, hata pale tunapojikumba chini na huzuni.

"…kulikuwa na njaa katika nchi."

Tusikuombe Mungu atujaze na amani na nguvu, ili tuweze kukaa imara katika majaribu yetu.

Jeremiah 29:9-13  (154 tokens)
Here's a short devotional in Swahili, following your instructions:

Mungu wetu ni mshindi wa upendo na amani! Hata pale tunapojikuta katika hali ngumu na wasiwasi, anatuahidi kwamba hayetuache wala kutuacha. Habari njema kwetu ni kwamba mipango yake kwa ajili yetu ni mizuri, yenye matumaini na yenye kusababisha mafanikio.

"Kwa maana ninajua mipango niliyo nayo kwa ajili yenu…"

Tubarikiwe, Mungu, kwa ahadi zako thabiti na tufanye mioyo yetu iwe wazi kukutafuta kwa moyo wote.

Matthew 3:16  (102 tokens)
Hii ni siku njema! Tunafurahia kuon

In [8]:
# Cell 6 (OPTIONAL) — one full Swahili STUDY GUIDE as a showcase artifact
import time
eng = ("They are telling you lies in my name; I have not sent them, says Yahweh. "
       "After seventy years in Babylon I will bring you back to this place. I know the plans "
       "I have for you — plans to prosper you and not to harm you, plans to give you hope and "
       "a future. Then you will call on me and come and pray to me, and I will listen to you. "
       "You will seek me and find me when you seek me with all your heart.")
swa = ("Wanawatabiria ninyi uongo kwa jina langu. Sikuwatuma, asema Bwana. Miaka sabini "
       "itakapotimia kwa ajili ya Babeli, nitakuja kwenu na kuwarudisha mahali hapa. Kwa maana "
       "ninajua mipango niliyo nayo kwa ajili yenu, ni mipango ya kuwafanikisha wala si ya "
       "kuwadhuru, ni mipango ya kuwapa tumaini katika siku zijazo. Kisha mtaniita, na mtakuja "
       "na kuniomba, nami nitawasikiliza. Mtanitafuta na kuniona, mtakaponitafuta kwa moyo wenu wote.")

prompt = (f"You are an experienced Bible teacher writing a Scripture STUDY GUIDE in SWAHILI ONLY, "
          f"for a general congregation. Base it on Jeremiah 29:9-13 below.\n"
          f"Use these section headings (translate them into Swahili) with Markdown '## ':\n"
          f"Passage, Background, Immediate Context, Explanation, Key Spiritual Themes, "
          f"Modern-Day Application, Reflection Questions (numbered), Closing Prayer.\n"
          f"Quote only the Swahili text when quoting. Write everything in Swahili.\n\n"
          f"English: {eng}\n\nSwahili: {swa}")

sp = SamplingParams(temperature=0.7, top_p=0.9, max_tokens=2048)
t0 = time.time()
out = llm.chat([[{"role": "user", "content": prompt}]], sp)
elapsed = time.time() - t0
text = out[0].outputs[0].text.strip()
ntok = len(out[0].outputs[0].token_ids)
print(text)
print(f"\n>>> {ntok} tokens in {elapsed:.1f}s = {ntok/elapsed:.1f} tokens/sec")
with open("swahili_study_guide.txt", "w", encoding="utf-8") as f:
    f.write(text + f"\n\n[{ntok} tokens, {ntok/elapsed:.1f} tok/s, gemma-3-12b-it on AMD gfx1100]\n")
print("Saved: swahili_study_guide.txt")

Processed prompts: 100%|██████████| 1/1 [01:32<00:00, 92.99s/it, est. speed input: 3.84 toks/s, output: 17.67 toks/s]

## Ufundisho wa Biblia: Yeremia 29:9-13

### Nukuu

"Wanawatabiria ninyi uongo kwa jina langu. Sikuwatuma, asema Bwana. Miaka sabini itakapotimia kwa ajili ya Babeli, nitakuja kwenu na kuwarudisha mahali hapa. Kwa maana ninajua mipango niliyo nayo kwa ajili yenu, ni mipango ya kuwafanikisha wala si ya kuwadhuru, ni mipango ya kuwapa tumaini katika siku zijazo. Kisha mtaniita, na mtakuja na kuniomba, nami nitawasikiliza. Mtanitafuta na kuniona, mtakaponitafuta kwa moyo wenu wote."

### Asili ya Uandishi (Background)

Yeremia alikuwa nabii muhimu katika ufalme wa Yuda kabla ya uharibifu wake na uhamisho wa Wayahudi kwenda Utumwa huko Babeli.  Alitabasamu (alionya) watu wake kuhusu hatari za kumtii Mungu na matokeo ya kutokutii.  Uhamisho huo ulikuwa matokeo ya kutokutii, na Yeremia alitumwa na Mungu kuwapa watu wake ujumbe wa faraja na tumaini wakati wa nyakati ngumu hizi.  Kitabu chote cha Yeremia kinajikita katika mahusiano ya Mungu na Israeli, na utabiri wake unaonyesha upendo na uvum